In [1]:
import sys
sys.path.append("/home/jiahuang/test-code-hazel/")
import gen_fxns
from plot_pca import plot_pca_loadings
import h5py
import numpy as np 
import matplotlib.pyplot as plt
from pathlib import Path
import scipy as sp
import sklearn
from datetime import date
import pandas as pd 
from datetime import timedelta
from scipy.stats import zscore
import seaborn as sns 
import pickle 
import os 
from datetime import datetime

In [4]:
bandref = dict({'delta':[3,4], 
                 'theta':[4,8], 
                 'alpha':[8,12], 
                 'beta':[12,30], 
                 'low_gamma':[30,50], 
                 'high_gamma':[70,110]})
ptIDs = ['RCS02', 'RCS03', 'RCS04', 'RCS05', 'RCS06', 'RCS07','RCS08','RCS09']
def build_long_df_zscored(pt, bandref, r2_threshold=0.9):
    record_path = f"/userdata/jiahuang/pain-data/Stage1-test/{pt}/records_df_fooof_{pt}.pkl"
    records_df = pd.read_pickle(record_path)
    records_df['roi'] = records_df['ch_label'].str.replace(r'_\d+$', '', regex=True)
    records_df = records_df[records_df['r2'] >= r2_threshold]

    freqs_welch = records_df['fooof'].iloc[0].freqs
    bands = list(bandref.keys())
    ch_labels = sorted(records_df['ch_label'].unique())
    rows = []
    thresh_low_aff = np.percentile(records_df['affective'], 30)
    thresh_high_aff = np.percentile(records_df['affective'], 70)

    thresh_low_sens = np.percentile(records_df['sensory'], 30)
    thresh_high_sens = np.percentile(records_df['sensory'], 70)


    for ch in ch_labels:
        sub = records_df[records_df['ch_label'] == ch].sort_values('trial_idx').reset_index(drop=True)
        
        corrected_stack = np.stack([r['fooof'].fooofed_spectrum_ - r['fooof']._ap_fit for _, r in sub.iterrows()])
        raw_stack       = np.stack([r['fooof'].power_spectrum for _, r in sub.iterrows()])  # trials x freq
        corrected_z = (corrected_stack - corrected_stack.mean(axis=0)) / corrected_stack.std(axis=0)
        raw_z       = (raw_stack       - raw_stack.mean(axis=0))       / raw_stack.std(axis=0)

        for i, row in sub.iterrows():
            for band in bands:
                fmin, fmax = bandref[band]
                idxf = np.where((freqs_welch >= fmin) & (freqs_welch <= fmax))[0]
                rows.append({
                    'ptID':             pt,
                    'trial_idx':        row['trial_idx'],
                    'ch_label':         ch,
                    'roi':              row['roi'],
                    'band':             band,
                    'band_power':       np.nanmean(corrected_stack[i, idxf]),
                    'band_power_z':     np.nanmean(corrected_z[i, idxf]),
                    'band_power_raw':   np.nanmean(raw_stack[i, idxf]),
                    'band_power_raw_z': np.nanmean(raw_z[i, idxf]),
                    'aff_score':        row['affective'],
                    'sens_score':       row['sensory'],
                    'stratified_group_aff':int(row['affective']>=thresh_high_aff) + int(row['affective']>=thresh_low_aff),
                    'stratified_group_sens':int(row['sensory']>=thresh_high_sens) + int(row['sensory']>=thresh_low_sens)
                })

    df = pd.DataFrame(rows)
    return df

In [7]:
single_pt_df = build_long_df_zscored('RCS02',bandref)

/tmp/ipykernel_3422188/1570667532.py:43: RuntimeWarning: Mean of empty slice
  'band_power':       np.nanmean(corrected_stack[i, idxf]),
/tmp/ipykernel_3422188/1570667532.py:44: RuntimeWarning: Mean of empty slice
  'band_power_z':     np.nanmean(corrected_z[i, idxf]),
/tmp/ipykernel_3422188/1570667532.py:45: RuntimeWarning: Mean of empty slice
  'band_power_raw':   np.nanmean(raw_stack[i, idxf]),
/tmp/ipykernel_3422188/1570667532.py:46: RuntimeWarning: Mean of empty slice
  'band_power_raw_z': np.nanmean(raw_z[i, idxf]),


In [22]:
single_pt_df['feat_col'] = single_pt_df['band'] + '__' + single_pt_df['ch_label']

scores = single_pt_df.drop_duplicates(['ptID', 'trial_idx'])[['ptID', 'trial_idx', 'aff_score', 'sens_score','stratified_group_aff','stratified_group_sens']]

# ---------------- df: channel-level (no aggregation, keep all channels) ----------------
ch_corrected = single_pt_df.pivot_table(
    index=['ptID', 'trial_idx'], columns='feat_col', values='band_power_z', aggfunc='mean'
)
ch_raw = single_pt_df.pivot_table(
    index=['ptID', 'trial_idx'], columns='feat_col', values='band_power_raw_z', aggfunc='mean'
)

ch_corrected.columns = [f"{c}_corrected_z" for c in ch_corrected.columns]
ch_raw.columns       = [f"{c}_raw_z" for c in ch_raw.columns]

feat_df_channel = pd.concat([ch_corrected, ch_raw], axis=1).reset_index()
feat_df_channel = feat_df_channel.merge(scores, on=['ptID', 'trial_idx'])

display(feat_df_channel)

,ptID,trial_idx,alpha__L AINS_1_corrected_z,alpha__L AINS_2_corrected_z,alpha__L AINS_3_corrected_z,alpha__L AINS_4_corrected_z,alpha__L IC_1_corrected_z,alpha__L IC_2_corrected_z,alpha__L IC_3_corrected_z,alpha__L IC_4_corrected_z,...,theta__R THAL_1_raw_z,theta__R THAL_2_raw_z,theta__R THAL_3_raw_z,theta__R dmPFC_1_raw_z,theta__R dmPFC_2_raw_z,theta__R dmPFC_3_raw_z,aff_score,sens_score,stratified_group_aff,stratified_group_sens
0,RCS02,32,-1.948326,-2.566986,-1.859643,-1.509631,1.079001,-1.146505,-2.259775,1.662312,...,2.763648,2.862411,2.953163,3.381103,5.655430,2.709581,-0.184285,-3.148555,1,0
1,RCS02,33,-1.948326,-2.605467,-2.713194,-1.350983,0.385482,0.331170,0.623713,0.228576,...,-0.473163,-0.439342,-0.427531,0.362515,0.269225,0.505575,-0.692122,-1.063422,0,0
2,RCS02,34,1.685963,-1.715783,-0.313363,-0.954368,-0.165465,-0.060427,-0.315242,0.361846,...,-0.511357,-0.391690,-0.437251,0.316232,0.165280,0.426411,0.541736,-0.576758,2,1
3,RCS02,35,1.291248,-0.156564,-0.073369,-0.112181,1.109511,0.146397,0.506845,0.404870,...,-0.469138,-0.419856,-0.417151,0.113757,0.194245,0.404756,5.890015,-0.225095,2,1
4,RCS02,36,1.240516,-0.164312,-0.536086,-0.126922,0.176644,-0.586517,0.454190,-0.560350,...,-0.330788,-0.222716,-0.400045,0.318228,0.035087,0.233424,-0.222151,0.962826,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,RCS02,113,-1.190050,-0.004532,-0.098929,1.018054,-1.068467,-0.146849,-0.091696,-0.914337,...,-0.548525,-0.428941,-0.515105,-0.574399,-0.712151,-0.551911,-0.876193,-1.812682,0,0
77,RCS02,114,-0.861066,0.064811,0.605654,0.590684,-0.629674,-0.894483,-0.337610,-0.194448,...,-0.568177,-0.560193,-0.511101,-0.278067,-0.111198,0.107030,-0.950385,-0.788074,0,0
78,RCS02,115,-0.138078,0.216510,0.797258,0.679800,-0.082354,0.148871,-0.309593,0.048458,...,-0.482769,-0.510108,-0.445222,-0.465778,-0.119317,0.274844,0.285652,-0.802229,2,0
79,RCS02,116,-0.643046,-0.559788,-0.358850,-0.613440,-0.868074,0.708859,-1.133042,0.499677,...,-0.468927,-0.449005,-0.404291,-1.126026,-1.047801,-1.504781,0.257412,-0.408101,1,1


In [23]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

ptID = 'RCS02'
X_df = feat_df_channel[feat_df_channel['ptID']==ptID]

y_df = scores[['aff_score', 'sens_score']].copy()

feat_cols = [c for c in X_df.columns if c.endswith('_corrected_z')]
print(len(feat_cols))
X_df = X_df[feat_cols].copy()

X_df = X_df.fillna(X_df.mean())

X = X_df.values

y_bins_aff = scores['stratified_group_aff']
y_bins_sens = scores['stratified_group_sens']
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ridge_cv = RidgeCV(alphas=[0.1, 0.5, 1, 2, 5, 10, 50, 100])

pipe = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('ridge', ridge_cv)])
y_pred_aff = cross_val_predict(pipe, X, scores['aff_score'], cv=skf.split(X, y_bins_aff))
y_pred_sens = cross_val_predict(pipe, X, scores['sens_score'], cv=skf.split(X, y_bins_sens))
print('R2 aff:', r2_score(scores['aff_score'], y_pred_aff))
print('R2 sens:', r2_score(scores['sens_score'], y_pred_sens))

190
R2 aff: 0.014339994622006857
R2 sens: 0.1535210832857271


In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer
import re
import numpy as np
import matplotlib.pyplot as plt

alphas_grid = [0.1, 0.5, 1, 2, 5, 10, 50, 100,1000]

# ---------------- 1. 用全部数据(该患者全部channel-trial行) + StratifiedKFold 选alpha + fit ----------------
imputer_full = SimpleImputer(strategy='mean').fit(X)
X_full_imputed = imputer_full.transform(X)

full_cv_aff  = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_full_imputed, y_bins_aff))
full_cv_sens = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_full_imputed, y_bins_sens))

final_ridge_aff  = RidgeCV(alphas=alphas_grid, cv=full_cv_aff).fit(X_full_imputed, scores['aff_score'])
final_ridge_sens = RidgeCV(alphas=alphas_grid, cv=full_cv_sens).fit(X_full_imputed, scores['sens_score'])

print(f"[{ptID}] final alpha_aff: {final_ridge_aff.alpha_}, final alpha_sens: {final_ridge_sens.alpha_}")

coef_df = pd.DataFrame({
    'coef_aff': final_ridge_aff.coef_,
    'coef_sens': final_ridge_sens.coef_,
}, index=X_df.columns)

# ---------------- 2. 拆列名 → band, roi ----------------
def parse_feat_name(name):
    band = name.split('__', 1)[0]
    roi = name.split('__', 1)[1]
    roi = re.sub(r'_corrected_z$', '', roi)
    return band, roi

parsed = coef_df.copy()
parsed[['band', 'channel']] = parsed.index.to_series().apply(lambda n: pd.Series(parse_feat_name(n)))

bands_order = ['delta', 'theta', 'alpha', 'beta', 'low_gamma', 'high_gamma']
bands_present = [b for b in bands_order if b in parsed['band'].unique()]
rois_present = sorted(parsed['channel'].unique())

mat_aff  = parsed.pivot_table(index='channel', columns='band', values='coef_aff').reindex(index=rois_present, columns=bands_present)
mat_sens = parsed.pivot_table(index='channel', columns='band', values='coef_sens').reindex(index=rois_present, columns=bands_present)

# ---------------- 3. 画heatmap，共享colorbar ----------------
max_abs = np.nanmax(np.abs(np.concatenate([mat_aff.values, mat_sens.values])))

fig, axes = plt.subplots(1, 2, figsize=(10, max(4, 0.4*len(rois_present))), sharey=True)

for ax, mat, title in zip(axes, [mat_aff, mat_sens], ['Affective', 'Sensory']):
    im = ax.imshow(mat.values, cmap='RdBu_r', vmin=-max_abs, vmax=max_abs, aspect='auto')
    ax.set_xticks(range(len(bands_present)))
    ax.set_xticklabels(bands_present, rotation=45, ha='right')
    ax.set_title(title)

axes[0].set_yticks(range(len(rois_present)))
axes[0].set_yticklabels(rois_present)
axes[0].set_ylabel('channel')

fig.colorbar(im, ax=axes, label='Ridge coefficient', shrink=0.8)
fig.suptitle(f'{ptID}: Ridge coefficients (channel-level, corrected_z): band x roi')
plt.show()